In [1]:
!pip install transformers datasets torch scikit-learn pandas numpy matplotlib seaborn -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME    = 'bert-base-uncased'
MAX_LEN       = 256
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
EPOCHS        = 3
NUM_LABELS    = 2
SAMPLE_SIZE   = 5000

print(f"Device: {DEVICE}")

C:\Users\Joshith\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


## 1. Data Preprocessing

In [3]:
raw = load_dataset('imdb')
df_train_full = pd.DataFrame(raw['train'])
df_test_full  = pd.DataFrame(raw['test'])

def preprocess_text(text):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r"[^a-zA-Z0-9\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip().lower()

def clean_df(df):
    df = df.copy()
    df.dropna(subset=['text', 'label'], inplace=True)
    df['text'] = df['text'].apply(preprocess_text)
    df = df[df['text'].str.len() > 10].reset_index(drop=True)
    return df

df_train_clean = clean_df(df_train_full)
df_test_clean  = clean_df(df_test_full)

print(f"Train: {len(df_train_clean):,} | Test: {len(df_test_clean):,}")
df_train_clean.head(3)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Generating unsupervised split: 100%|██████████████████████████████████| 50000/50000 [00:00<00:00, 217586.39 examples/s]


Train: 25,000 | Test: 25,000


,text,label
0,i rented i am curious yellow from my video sto...,0
1,i am curious yellow is a risible and pretentio...,0
2,if only to avoid making this type of film in t...,0


## 2. Data Splitting

In [4]:
df_sample, _ = train_test_split(df_train_clean, train_size=SAMPLE_SIZE, stratify=df_train_clean['label'], random_state=SEED)
df_train, df_val = train_test_split(df_sample, test_size=0.2, stratify=df_sample['label'], random_state=SEED)
df_test, _ = train_test_split(df_test_clean, train_size=1000, stratify=df_test_clean['label'], random_state=SEED)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

Train: 4000 | Val: 1000 | Test: 1000


## 3. Tokenization

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IMDBDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(str(self.texts[idx]), max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'token_type_ids' : enc['token_type_ids'].squeeze(0),
            'labels'         : torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

train_loader = DataLoader(IMDBDataset(df_train['text'], df_train['label']), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(IMDBDataset(df_val['text'],   df_val['label']),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(IMDBDataset(df_test['text'],  df_test['label']),  batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

Train batches: 250 | Val batches: 63 | Test batches: 63


## 4. Model Building & Fine-Tuning Utilities

In [6]:
def build_model():
    return AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(DEVICE)

def freeze_all(model):
    for name, p in model.named_parameters():
        p.requires_grad = 'classifier' in name
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def freeze_except_last2(model):
    for p in model.parameters():
        p.requires_grad = False
    for i in [10, 11]:
        for p in model.bert.encoder.layer[i].parameters():
            p.requires_grad = True
    for p in model.bert.pooler.parameters():
        p.requires_grad = True
    for p in model.classifier.parameters():
        p.requires_grad = True
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True
    print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        ttype  = batch['token_type_ids'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        out = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype, labels=labels)
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += out.loss.item()
        preds = torch.argmax(out.logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    for batch in loader:
        ids    = batch['input_ids'].to(DEVICE)
        mask   = batch['attention_mask'].to(DEVICE)
        ttype  = batch['token_type_ids'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        out    = model(input_ids=ids, attention_mask=mask, token_type_ids=ttype, labels=labels)
        preds  = torch.argmax(out.logits, dim=1)
        total_loss += out.loss.item()
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), correct / total, np.array(all_preds), np.array(all_labels)

def get_metrics(y_true, y_pred):
    return {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall'   : recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1'       : f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative','Positive'], yticklabels=['Negative','Positive'])
    plt.title(title); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.tight_layout(); plt.show()

def run_experiment(name, freeze_fn):
    print(f"\n{'='*55}\n{name}\n{'='*55}")
    model = build_model()
    freeze_fn(model)
    optimizer    = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=0.01)
    total_steps  = len(train_loader) * EPOCHS
    scheduler    = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
    history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc      = train_epoch(model, train_loader, optimizer, scheduler)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader)
        history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss);   history['val_acc'].append(vl_acc)
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f} | Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}")

    _, _, y_pred, y_true = evaluate(model, test_loader)
    metrics = get_metrics(y_true, y_pred)
    print(f"\nTest Results: {metrics}")
    print(classification_report(y_true, y_pred, target_names=['Negative','Positive']))
    plot_cm(y_true, y_pred, title=name)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ep = range(1, EPOCHS + 1)
    ax[0].plot(ep, history['train_loss'], label='Train'); ax[0].plot(ep, history['val_loss'], label='Val')
    ax[0].set_title('Loss'); ax[0].legend()
    ax[1].plot(ep, history['train_acc'], label='Train'); ax[1].plot(ep, history['val_acc'], label='Val')
    ax[1].set_title('Accuracy'); ax[1].legend()
    plt.suptitle(name); plt.tight_layout(); plt.show()

    return {'name': name, 'metrics': metrics, 'history': history, 'y_pred': y_pred, 'y_true': y_true}

## 5. Evaluation

In [ ]:
all_results = []

for name, freeze_fn in [
    ('Experiment 1: Frozen BERT (Classifier Only)', freeze_all),
    ('Experiment 2: Fine-Tune Last 2 Layers',       freeze_except_last2),
    ('Experiment 3: Full Fine-Tuning',              unfreeze_all),
]:
    all_results.append(run_experiment(name, freeze_fn))

r1, r2, r3 = all_results

rows = []
for r in all_results:
    row = {'Experiment': r['name']}
    row.update({k.capitalize(): round(v, 4) for k, v in r['metrics'].items()})
    rows.append(row)

df_cmp = pd.DataFrame(rows).set_index('Experiment')
print(df_cmp.to_string())

metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1']
x     = np.arange(len(metrics_list))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, (r, label) in enumerate(zip(all_results, ['Exp 1', 'Exp 2', 'Exp 3'])):
    vals = [r['metrics'][m.lower()] for m in metrics_list]
    bars = ax.bar(x + i * width, vals, width, label=label)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{v:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_list)
ax.set_ylim(0, 1.1)
ax.set_title('Experiment Comparison')
ax.legend()
plt.tight_layout()
plt.show()


Experiment 1: Frozen BERT (Classifier Only)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 1,538
